# **entire rag pipeline**

## **import**

In [1]:
import os
import json
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import  BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from groq import Groq
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from langchain_cohere import CohereRerank
from dotenv import load_dotenv
load_dotenv()

C:\Users\Admin\AppData\Local\Temp\ipykernel_10848\265032013.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, DirectoryLoader
c:\Users\Admin\Desktop\RAG-learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### **we're gonna deal with two kinds of documents, first simple textual document and then a pdf**

In [2]:
langchain_docs = []

### **1-text**

In [3]:
## loading the documents

text_data_path = "docs"
if not os.path.exists(text_data_path):
    raise FileNotFoundError("no textual data found")
text_loader = DirectoryLoader(
    path=text_data_path,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
text_docs = text_loader.load()

##chunking

chunk_size = 1000
chunk_overlap = 0
textsplitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap = chunk_overlap
)

text_chunks = textsplitter.split_documents(text_docs)
text_langchain_chunks = [
    Document(page_content=chunk.page_content,
              metadata={
                             "original_content":json.dumps(
                                 {
                                 "raw_text": chunk.page_content,
                                 
                             }
                             )
                         }
                     ) for chunk in text_chunks
             ] 

langchain_docs.extend(text_langchain_chunks)


### **2-pdf**

In [6]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)


## loading the elements of the pdf
print("--- loading pdf elements ---")
pdf_file_path = "./docs/attention-is-all-you-need_jusqua_results.pdf"
pdf_elements = partition_pdf(
    filename=pdf_file_path,
    strategy="hi_res",
    infer_table_structure=True,
    extract_image_block_types=["Image"],
    extract_image_block_to_payload=True
)

##chunking by title
pdf_chunks = chunk_by_title(
    pdf_elements,
    max_characters=3000,
    new_after_n_chars=2400,
    combine_text_under_n_chars=500
)

##seperating chunk content
pdf_chunks_with_content = []
for pdf_chunk in pdf_chunks:
    chunk_content = {
        "raw_text":pdf_chunk.text,
        "raw_tables":[],
        "raw_images":[]
    }
    if hasattr(pdf_chunk,"metadata") and hasattr(pdf_chunk.metadata,"orig_elements"):
        for ele in pdf_chunk.metadata.orig_elements:
            ele_type = type(ele).__name__
            if ele_type == "Table":
                html_table = getattr(ele,"text_as_html",ele.text)
                chunk_content["raw_tables"].append(html_table)
            elif ele_type == "Image":
                image_b64 = getattr(ele.metadata,"image_base64",ele.text)
                chunk_content["raw_images"].append(image_b64)
    pdf_chunks_with_content.append(chunk_content)


## adding ai summary for images and tables
print("creating summary of images and tables")

for pdf_chunk in pdf_chunks_with_content:
    if pdf_chunk["raw_tables"] or pdf_chunk["raw_images"]:
        prompt = f"""
                You are creating a searchable description for document content retrieval.
        
                CONTENT TO ANALYZE:
                TEXT CONTENT: {pdf_chunk["raw_text"]}
        
                    """
        for i,table in enumerate(pdf_chunk["raw_tables"]):
             prompt += f"""
                table {i+1} : {table}

                """
        
        prompt += """
            YOUR TASK:
                            Generate a comprehensive, searchable description that covers:
            
                            1. Key facts, numbers, and data points from text and tables
                            2. Main topics and concepts discussed  
                            3. Questions this content could answer
                            4. Visual content analysis (charts, diagrams, patterns in images)
                            5. Alternative search terms users might use
            
                            Make it detailed and searchable - prioritize findability over brevity.
            
                            SEARCHABLE DESCRIPTION:
            """    
        final_prompt = [{
            "type":"text",
            "text":prompt
        }]
        for i,image in enumerate(pdf_chunk["raw_images"]):
            final_prompt.append({
                            "type":"image_url",
                            "image_url":{"url":f"data:image/jpeg;base64,{image}"}
                        })
        messages = [
            {"role":"user",
             "content":final_prompt
            }
        ]
        response = client.chat.completions.create(
            model="qwen/qwen3.6-27b",
            messages=messages,
            temperature=0
        )
        pdf_chunk["summary"] = response.choices[0].message.content

        ## creating the langchain documents
pdf_langchain = []
for pdf_chunk in pdf_chunks_with_content:
    if pdf_chunk.get("summary"):
        pdf_doc = Document(page_content=pdf_chunk["summary"],
                           metadata={
                                                        "original_content":json.dumps(
                                                            {
                                                            "raw_text": pdf_chunk["raw_text"],
                                                            "raw_tables":pdf_chunk["raw_tables"],
                                                            "raw_images":pdf_chunk["raw_images"]
                                                            
                                                        }
                                                        )
                                                    })
        pdf_langchain.append(pdf_doc)
    else:
        pdf_langchain.append(Document(page_content=pdf_chunk["raw_text"],
                                metadata={
                                        "original_content":json.dumps(
                                         {
                                         "raw_text": pdf_chunk["raw_text"], })}))
langchain_docs.extend(pdf_langchain)

print("--- finished creating pdf chunks ---")

     

--- loading pdf elements ---
creating summary of images and tables
--- finished creating pdf chunks ---


## **storing in vectore db**

In [7]:
dir = "db/entire_pipeline_db"
print("--- creating db ---")
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)
vector_store = Chroma.from_documents(
    documents=langchain_docs,
    embedding=embedding_model,
    persist_directory=dir,
    collection_metadata={"hnsw:space":"cosine"}
)

print("--- finished creating db ---")

--- creating db ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4516.72it/s]


--- finished creating db ---


## **creating the retrievers**

In [8]:
#vector retriever
vector_retriever = vector_store.as_retriever(search_kwargs={"k":10})

In [9]:
#key words retriever
keyword_retriever = BM25Retriever.from_documents(langchain_docs)
keyword_retriever.k = 10

### **hybrid retriever**

In [10]:
ensemble_retriever = EnsembleRetriever(retrievers=[vector_retriever,keyword_retriever],weights=[0.7,0.3])

## **multi query**

In [19]:
print("--- creating the multi query ---")
original_query = "what was the complexity layer  of self attention layer type"
query_prompt = f"""Generate 3 different variations of this query that would help retrieve relevant documents:

Original query: {original_query}

Return 3 alternative queries that rephrase or approach the same question from different angles."""
query_messages = [{
    "role":"user",
    "content":query_prompt
}]
query_response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0,
    messages=query_messages,
    response_format={
        "type":"json_schema",
        "json_schema":{
            "name":"questions",
            "strict":True,
            "schema":{
                "type":"object",
                "properties":{
                    "questions":{
                        "type":"array",
                        "items":{
                            "type":"string"
                        }
                    }
                },
                "required":["questions"],
                "additionalProperties":False
            }
        }
    }

)
data = json.loads(query_response.choices[0].message.content)
questions = data["questions"]
questions.append(original_query)
print(questions)

--- creating the multi query ---
['What is the computational complexity of a self-attention layer?', 'How complex is the self-attention layer in terms of time and space complexity?', 'What is the complexity class of the self-attention mechanism layer?', 'what was the complexity layer  of self attention layer type']


## **retrieving**

In [20]:
print("--- retrieving all the docs from different questions ---")
all_retrievel_results = []
for question in questions:
    docs = ensemble_retriever.invoke(question)
    all_retrievel_results.append(docs)

--- retrieving all the docs from different questions ---


## **RRF**

In [21]:
print("--- applying RRF ---")
results_chunks = {}

k = 60
for chunk_list in all_retrievel_results:
    for position,chunk in enumerate(chunk_list):
        
        entry = results_chunks.setdefault(chunk.page_content,{"score":0,"doc":chunk})
        entry["score"] += 1 / (k + position)
sorted_docs = [v["doc"] for v in sorted(results_chunks.values(), key=lambda x: x["score"], reverse=True)]

--- applying RRF ---


## **reranker**

In [22]:
print("--- reranking ---")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
reranker = CohereRerank(model="rerank-english-v3.0",top_n=10,cohere_api_key=COHERE_API_KEY)
reranked_chunks = reranker.compress_documents(sorted_docs,original_query)

--- reranking ---


## **LLM**

In [24]:
llm_prompt_text = f"""Based on the following documents, please answer this question: {original_query}

CONTENT TO ANALYZE:
"""

for i, chunk in enumerate(reranked_chunks[:3]):
            llm_prompt_text += f"--- Document {i+1} ---\n"
                
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                    
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    llm_prompt_text += f"TEXT:\n{raw_text}\n\n"
                    
                # Add tables as HTML
                tables_html = original_data.get("raw_tables", [])
                if tables_html:
                    llm_prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        llm_prompt_text += f"Table {j+1}:\n{table}\n\n"
                
            llm_prompt_text += "\n"
            
llm_prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents.
return just the answer to the question"
 ANSWER:"""

llm_message_content = [{
      "type":"text",
      "text":llm_prompt_text
}]
 # Add all images from all chunks
for i,chunk in enumerate(reranked_chunks):
    if "original_content" in chunk.metadata:
        original_data = json.loads(chunk.metadata["original_content"])
        images_base64 = original_data.get("raw_images", [])
                
        for j,image_base64 in enumerate(images_base64):
                llm_message_content.append({
                "type": "text",
                "text": f"Image {j+1} of Document {i+1}:"
                })
                llm_message_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
final_llm_prompt = [
      {
            "role":"user",
            "content":llm_message_content
      }
]

response = client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=final_llm_prompt,
    temperature=0
    )
print(response.choices[0].message.content)


<think>
The user is asking for the complexity of the self-attention layer type based on the provided documents.

1.  **Scan Document 1:** I see "Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations for different layer types."
2.  **Locate "Self-Attention" in Table 1:** The first row under "Layer Type" is "Self-Attention".
3.  **Identify the "Complexity per Layer" column:** The column header is "Complexity per Layer".
4.  **Extract the value:** The value corresponding to "Self-Attention" in the "Complexity per Layer" column is "O(n^2 · d)".
5.  **Verify with text:** In Document 2, section "4 Why Self-Attention", it mentions "In terms of computational complexity, self-attention layers are faster than recurrent layers when the sequence length n is smaller than the representation dimensionality d...". It doesn't explicitly state the formula again in the text body as clearly as the table, but the table is definitive.
6.  **Formulate the answer:** 